## 1. Imports

In [41]:
import os
import re
import pickle
import warnings

import numpy as np
import pandas as pd

import spacy

from tqdm.auto import tqdm

from spacy.matcher import PhraseMatcher

from spacy.pipeline import EntityRuler

warnings.filterwarnings("ignore")

tqdm.pandas()

## 2. Paths

In [42]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

ENTITY_PATH = os.path.join(
    PROCESSED_PATH,
    "entities"
)

FINANCIAL_DATASET = os.path.join(
    PROCESSED_PATH,
    "financial_intelligence",
    "financial_intelligence_dataset.parquet"
)

os.makedirs(
    ENTITY_PATH,
    exist_ok=True
)

## 3. Load Dataset

In [43]:
financial_df = pd.read_parquet(
    FINANCIAL_DATASET
)

print("Financial Dataset")

print(financial_df.shape)

display(financial_df.head())

Financial Dataset
(3215296, 22)


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
0,1,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 14:30:54+00:00,A,analyst,2020,6,5,...,39,7,True,Stocks That Hit 52-Week Highs On Friday,39,7,Market Movement,Neutral,1.0,Rule
1,2,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 14:45:20+00:00,A,analyst,2020,6,3,...,42,7,True,Stocks That Hit 52-Week Highs On Wednesday,42,7,Market Movement,Neutral,1.0,Rule
2,3,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 08:30:07+00:00,A,analyst,2020,5,26,...,29,5,True,71 Biggest Movers From Friday,29,5,Market Movement,Neutral,1.0,Rule
3,4,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 16:45:06+00:00,A,analyst,2020,5,22,...,44,7,True,46 Stocks Moving In Friday's Mid-Day Session,44,7,Market Movement,Neutral,1.0,Rule
4,5,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 15:38:59+00:00,A,analyst,2020,5,22,...,87,14,True,B of A Securities Maintains Neutral on Agilent...,87,14,Analyst Rating,Bullish,1.0,Rule


## 4. Load spaCy

In [44]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [45]:
nlp = spacy.load(

    "en_core_web_sm",

    disable=[

        "parser",

        "lemmatizer",

        "textcat"

    ]

)

print(nlp.pipe_names)

['tok2vec', 'tagger', 'attribute_ruler', 'ner']


## 5. Financial Entity Dictionary

In [46]:
FINANCIAL_ENTITY_DICTIONARY = {

    "FINANCIAL_METRIC": [

        "revenue",
        "sales",
        "earnings",
        "profit",
        "net income",
        "operating income",
        "gross profit",
        "gross margin",
        "operating margin",
        "ebit",
        "ebitda",
        "eps",
        "cash flow",
        "free cash flow",
        "fcf",
        "book value",
        "market capitalization",
        "market cap",
        "valuation",
        "guidance",
        "forecast",
        "outlook",
        "price target"

    ],

    "CORPORATE_ACTION": [

        "ipo",
        "initial public offering",
        "merger",
        "acquisition",
        "buyout",
        "takeover",
        "joint venture",
        "partnership",
        "collaboration",
        "dividend",
        "stock split",
        "reverse stock split",
        "share buyback",
        "buyback",
        "spin-off",
        "spinoff",
        "delisting",
        "bankruptcy",
        "restructuring",
        "layoffs",
        "chapter 11"

    ],

    "ANALYST_ACTION": [

        "upgrade",
        "downgrade",
        "buy",
        "sell",
        "hold",
        "outperform",
        "underperform",
        "overweight",
        "underweight",
        "neutral",
        "strong buy",
        "equal weight",
        "price target",
        "maintains",
        "raises target",
        "cuts target"

    ],

    "SECURITY": [

        "stock",
        "share",
        "bond",
        "etf",
        "adr",
        "future",
        "futures",
        "option",
        "call option",
        "put option",
        "warrant",
        "preferred stock"

    ],

    "MARKET_INDEX": [

        "s&p 500",
        "nasdaq",
        "dow jones",
        "russell 2000",
        "ftse",
        "nikkei",
        "sensex",
        "nifty",
        "hang seng"

    ],

    "REGULATOR": [

        "sec",
        "fda",
        "federal reserve",
        "fed",
        "ecb",
        "doj",
        "department of justice",
        "cftc",
        "finra",
        "irs"

    ],

    "CRYPTO": [

        "bitcoin",
        "ethereum",
        "dogecoin",
        "litecoin",
        "ripple",
        "blockchain",
        "stablecoin",
        "crypto"

    ]

}

print("Categories :", len(FINANCIAL_ENTITY_DICTIONARY))

for category, terms in FINANCIAL_ENTITY_DICTIONARY.items():

    print(category, ":", len(terms))

Categories : 7
FINANCIAL_METRIC : 23
CORPORATE_ACTION : 21
ANALYST_ACTION : 16
SECURITY : 12
MARKET_INDEX : 9
REGULATOR : 10
CRYPTO : 8


## 6. Quarter Dictionary

In [47]:
QUARTERS = [

    "Q1",
    "Q2",
    "Q3",
    "Q4",

    "first quarter",
    "second quarter",
    "third quarter",
    "fourth quarter",

    "fiscal year",
    "fy",
    "fy20",
    "fy21",
    "fy22",
    "fy23",
    "fy24",
    "fy25",
    "fy26"

]

## 7. Currency Dictionary

In [48]:
CURRENCIES = [

    "usd",
    "eur",
    "gbp",
    "cad",
    "aud",
    "jpy",
    "cny",
    "inr",
    "rupee",
    "dollar",
    "euro",
    "yen",
    "pound"

]

## 8. Build PhraseMatcher

In [49]:
matcher = PhraseMatcher(

    nlp.vocab,

    attr="LOWER"

)

for label, phrases in FINANCIAL_ENTITY_DICTIONARY.items():

    patterns = [

        nlp.make_doc(text)

        for text in phrases

    ]

    matcher.add(

        label,

        patterns

    )

matcher.add(

    "QUARTER",

    [

        nlp.make_doc(text)

        for text in QUARTERS

    ]

)

matcher.add(

    "CURRENCY",

    [

        nlp.make_doc(text)

        for text in CURRENCIES

    ]

)

print("Number of Matcher Labels:", len(matcher))

Number of Matcher Labels: 9


## 9. Add EntityRuler

In [50]:
ruler = nlp.add_pipe(

    "entity_ruler",

    before="ner"

)

patterns = []

for label, phrases in FINANCIAL_ENTITY_DICTIONARY.items():

    for phrase in phrases:

        patterns.append({

            "label": label,

            "pattern": phrase

        })

for phrase in QUARTERS:

    patterns.append({

        "label": "QUARTER",

        "pattern": phrase

    })

for phrase in CURRENCIES:

    patterns.append({

        "label": "CURRENCY",

        "pattern": phrase

    })

ruler.add_patterns(patterns)

print("EntityRuler Patterns :", len(patterns))

EntityRuler Patterns : 129


## 10. Financial Regex Patterns

In [51]:
REGEX_PATTERNS = {

    "MONEY": re.compile(
        r"\$\s?\d+(?:\.\d+)?\s?(?:million|billion|trillion|m|bn|b)?",
        re.IGNORECASE
    ),

    "PERCENTAGE": re.compile(
        r"\d+(?:\.\d+)?%",
        re.IGNORECASE
    ),

    "EPS": re.compile(
        r"(?:eps|earnings per share)\s*(?:of|was|is|:)?\s*\$?\d+(?:\.\d+)?",
        re.IGNORECASE
    ),

    "REVENUE": re.compile(
        r"revenue\s*(?:of|was|is|:)?\s*\$?\d+(?:\.\d+)?\s?(?:million|billion|m|bn|b)?",
        re.IGNORECASE
    ),

    "EBITDA": re.compile(
        r"ebitda\s*(?:of|was|is|:)?\s*\$?\d+(?:\.\d+)?",
        re.IGNORECASE
    ),

    "QUARTER": re.compile(
        r"\bQ[1-4]\b(?:\s?\d{4})?",
        re.IGNORECASE
    ),

    "FISCAL_YEAR": re.compile(
        r"\bFY\s?\d{2,4}\b|\bFiscal Year\b",
        re.IGNORECASE
    ),

    "TICKER": re.compile(
        r"\((NASDAQ|NYSE|AMEX|OTC):\s?[A-Z]{1,5}\)",
        re.IGNORECASE
    ),

    "PRICE_TARGET": re.compile(
        r"price target\s*(?:of|to|at)?\s*\$?\d+(?:\.\d+)?",
        re.IGNORECASE
    )

}

print("Regex Categories :", len(REGEX_PATTERNS))

Regex Categories : 9


## 11. Regex Extraction Function

In [52]:
def extract_regex_entities(text):

    entities = []

    if pd.isna(text):

        return entities

    text = str(text)

    for label, pattern in REGEX_PATTERNS.items():

        matches = pattern.finditer(text)

        for match in matches:

            entities.append({

                "entity": match.group(),

                "label": label,

                "source": "Regex",

                "confidence": 1.0

            })

    return entities

## 12. Hybrid Entity Extraction

In [53]:
def extract_entities(text):

    if pd.isna(text):

        return []

    doc = nlp(str(text))

    entities = []

    seen = set()

    # spaCy entities
    for ent in doc.ents:

        key = (

            ent.text.lower(),

            ent.label_

        )

        if key not in seen:

            seen.add(key)

            entities.append({

                "entity": ent.text,

                "label": ent.label_,

                "source": "spaCy",

                "confidence": 0.95

            })

    # PhraseMatcher
    matches = matcher(doc)

    for match_id, start, end in matches:

        span = doc[start:end]

        label = nlp.vocab.strings[match_id]

        key = (

            span.text.lower(),

            label

        )

        if key not in seen:

            seen.add(key)

            entities.append({

                "entity": span.text,

                "label": label,

                "source": "Matcher",

                "confidence": 1.0

            })

    # Regex
    regex_entities = extract_regex_entities(text)

    for item in regex_entities:

        key = (

            item["entity"].lower(),

            item["label"]

        )

        if key not in seen:

            seen.add(key)

            entities.append(item)

    return entities

## 13. Test Extraction

In [54]:
sample_headlines = [

    "Apple reports Q2 revenue of $98 billion and EPS of $2.35.",

    "Microsoft acquires Activision for $68.7 billion.",

    "Tesla announces stock buyback program.",

    "Federal Reserve raises interest rates by 0.25%.",

    "NVIDIA receives price target of $175 from Morgan Stanley."

]

for headline in sample_headlines:

    print()

    print(headline)

    print("-" * 60)

    entities = extract_entities(headline)

    for entity in entities:

        print(entity)


Apple reports Q2 revenue of $98 billion and EPS of $2.35.
------------------------------------------------------------
{'entity': 'Apple', 'label': 'ORG', 'source': 'spaCy', 'confidence': 0.95}
{'entity': 'Q2', 'label': 'QUARTER', 'source': 'spaCy', 'confidence': 0.95}
{'entity': 'revenue', 'label': 'FINANCIAL_METRIC', 'source': 'spaCy', 'confidence': 0.95}
{'entity': '$98 billion', 'label': 'MONEY', 'source': 'spaCy', 'confidence': 0.95}
{'entity': 'EPS', 'label': 'ORG', 'source': 'spaCy', 'confidence': 0.95}
{'entity': '2.35', 'label': 'MONEY', 'source': 'spaCy', 'confidence': 0.95}
{'entity': 'EPS', 'label': 'FINANCIAL_METRIC', 'source': 'Matcher', 'confidence': 1.0}
{'entity': '$2.35', 'label': 'MONEY', 'source': 'Regex', 'confidence': 1.0}
{'entity': 'EPS of $2.35', 'label': 'EPS', 'source': 'Regex', 'confidence': 1.0}
{'entity': 'revenue of $98 billion', 'label': 'REVENUE', 'source': 'Regex', 'confidence': 1.0}

Microsoft acquires Activision for $68.7 billion.
------------------

## 14. Batch Processing Configuration

In [55]:
unique_headlines = (

    financial_df

    .drop_duplicates(

        subset="headline"

    )

    .reset_index(drop=True)

)

TOTAL_HEADLINES = len(unique_headlines)

BATCH_SIZE = 50000

OUTPUT_BATCH_DIR = os.path.join(

    ENTITY_PATH,

    "entity_batches"

)

os.makedirs(

    OUTPUT_BATCH_DIR,

    exist_ok=True

)

TOTAL_BATCHES = (

    TOTAL_HEADLINES + BATCH_SIZE - 1

) // BATCH_SIZE

print("Original Headlines :", len(financial_df))
print("Unique Headlines   :", TOTAL_HEADLINES)
print("Duplicates Removed :", len(financial_df) - TOTAL_HEADLINES)
print("Batch Size         :", BATCH_SIZE)
print("Total Batches      :", TOTAL_BATCHES)

Original Headlines : 3215296
Unique Headlines   : 1649929
Duplicates Removed : 1565367
Batch Size         : 50000
Total Batches      : 33


## 15. Process One Batch

In [56]:
def process_entity_batch(df_batch):

    rows = []

    for row in df_batch.itertuples(index=False):

        entities = extract_entities(row.headline)

        for entity in entities:

            rows.append({

                "news_id": row.news_id,

                "ticker": row.ticker,

                "published_date": row.published_date,

                "headline": row.headline,

                "entity": entity["entity"],

                "entity_label": entity["label"],

                "entity_source": entity["source"],

                "confidence": entity["confidence"]

            })

    return pd.DataFrame(rows)

## 16. Test One Batch

In [57]:
test_batch = unique_headlines.head(1000)

entity_test = process_entity_batch(test_batch)

print(entity_test.shape)

display(entity_test.head())

(3718, 8)


,news_id,ticker,published_date,headline,entity,entity_label,entity_source,confidence
0,1,A,2020-06-05 14:30:54+00:00,Stocks That Hit 52-Week Highs On Friday,52-Week,DATE,spaCy,0.95
1,1,A,2020-06-05 14:30:54+00:00,Stocks That Hit 52-Week Highs On Friday,Friday,DATE,spaCy,0.95
2,2,A,2020-06-03 14:45:20+00:00,Stocks That Hit 52-Week Highs On Wednesday,52-Week,DATE,spaCy,0.95
3,2,A,2020-06-03 14:45:20+00:00,Stocks That Hit 52-Week Highs On Wednesday,Wednesday,DATE,spaCy,0.95
4,3,A,2020-05-26 08:30:07+00:00,71 Biggest Movers From Friday,71,CARDINAL,spaCy,0.95


## 17. Batch Processing Loop

In [58]:
print("Original headlines :", len(financial_df))
print("Unique headlines   :", financial_df["headline"].nunique())

duplicate_rate = (
    1 - financial_df["headline"].nunique() / len(financial_df)
) * 100

print(f"Duplicate rate : {duplicate_rate:.2f}%")

Original headlines : 3215296
Unique headlines   : 1649929
Duplicate rate : 48.69%


In [59]:
# for batch_no in tqdm(range(TOTAL_BATCHES)):

#     start = batch_no * BATCH_SIZE

#     end = min(

#         start + BATCH_SIZE,

#         TOTAL_HEADLINES

#     )

#     batch = unique_headlines.iloc[start:end]

#     entity_batch = process_entity_batch(batch)

#     output_file = os.path.join(

#         OUTPUT_BATCH_DIR,

#         f"entity_batch_{batch_no:03d}.parquet"

#     )

#     entity_batch.to_parquet(

#         output_file,

#         index=False

#     )

In [60]:
import glob

entity_files = sorted(

    glob.glob(

        os.path.join(

            OUTPUT_BATCH_DIR,

            "*.parquet"

        )

    )

)

entity_df = pd.concat(

    [

        pd.read_parquet(f)

        for f in entity_files

    ],

    ignore_index=True

)

print(entity_df.shape)

display(entity_df.head())

(5851032, 8)


,news_id,ticker,published_date,headline,entity,entity_label,entity_source,confidence
0,1,A,2020-06-05 14:30:54+00:00,Stocks That Hit 52-Week Highs On Friday,52-Week,DATE,spaCy,0.95
1,1,A,2020-06-05 14:30:54+00:00,Stocks That Hit 52-Week Highs On Friday,Friday,DATE,spaCy,0.95
2,2,A,2020-06-03 14:45:20+00:00,Stocks That Hit 52-Week Highs On Wednesday,52-Week,DATE,spaCy,0.95
3,2,A,2020-06-03 14:45:20+00:00,Stocks That Hit 52-Week Highs On Wednesday,Wednesday,DATE,spaCy,0.95
4,3,A,2020-05-26 08:30:07+00:00,71 Biggest Movers From Friday,71,CARDINAL,spaCy,0.95


In [61]:
headline_entities = (

    entity_df

    .groupby("headline")

    .agg({

        "entity": list,

        "entity_label": list,

        "entity_source": list

    })

    .reset_index()

)

In [62]:
financial_df = financial_df.merge(

    headline_entities,

    on="headline",

    how="left"

)

print(financial_df.shape)

display(financial_df.head())

(3215296, 25)


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source,entity,entity_label,entity_source
0,1,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 14:30:54+00:00,A,analyst,2020,6,5,...,Stocks That Hit 52-Week Highs On Friday,39,7,Market Movement,Neutral,1.0,Rule,"[52-Week, Friday]","[DATE, DATE]","[spaCy, spaCy]"
1,2,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 14:45:20+00:00,A,analyst,2020,6,3,...,Stocks That Hit 52-Week Highs On Wednesday,42,7,Market Movement,Neutral,1.0,Rule,"[52-Week, Wednesday]","[DATE, DATE]","[spaCy, spaCy]"
2,3,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 08:30:07+00:00,A,analyst,2020,5,26,...,71 Biggest Movers From Friday,29,5,Market Movement,Neutral,1.0,Rule,"[71, Biggest Movers, Friday]","[CARDINAL, ORG, DATE]","[spaCy, spaCy, spaCy]"
3,4,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 16:45:06+00:00,A,analyst,2020,5,22,...,46 Stocks Moving In Friday's Mid-Day Session,44,7,Market Movement,Neutral,1.0,Rule,"[46, Friday, Mid-Day Session]","[CARDINAL, DATE, LOC]","[spaCy, spaCy, spaCy]"
4,5,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 15:38:59+00:00,A,analyst,2020,5,22,...,B of A Securities Maintains Neutral on Agilent...,87,14,Analyst Rating,Bullish,1.0,Rule,"[88, Maintains, Neutral, Price Target, Price T...","[MONEY, ANALYST_ACTION, ANALYST_ACTION, FINANC...","[spaCy, Matcher, Matcher, Matcher, Matcher, Re..."


In [63]:
import gc

del entity_df
del headline_entities
del unique_headlines
del entity_files

gc.collect()

0

## 18. Entity Normalization

In [64]:
ENTITY_NORMALIZATION = {

    "apple inc": "Apple",
    "apple": "Apple",
    "aapl": "Apple",

    "microsoft": "Microsoft",
    "msft": "Microsoft",

    "alphabet": "Google",
    "google": "Google",
    "goog": "Google",
    "googl": "Google",

    "amazon": "Amazon",
    "amzn": "Amazon",

    "tesla": "Tesla",
    "tsla": "Tesla",

    "nvidia": "NVIDIA",
    "nvda": "NVIDIA",

    "meta": "Meta",
    "facebook": "Meta",

    "netflix": "Netflix",
    "nflx": "Netflix"

}

## 19. Normalization Function

In [65]:
def normalize_entity(entity):

    if pd.isna(entity):

        return entity

    entity = str(entity).strip()

    key = entity.lower()

    if key in ENTITY_NORMALIZATION:

        return ENTITY_NORMALIZATION[key]

    return entity

## 20. Normalize All Entities

In [66]:
financial_df["normalized_entities"] = (

    financial_df["entity"]

    .apply(

        lambda x: [

            normalize_entity(e)

            for e in x

        ]

        if isinstance(x, list)

        else []

    )

)

display(

    financial_df[

        [

            "entity",

            "normalized_entities"

        ]

    ].head()

)

,entity,normalized_entities
0,"[52-Week, Friday]","[52-Week, Friday]"
1,"[52-Week, Wednesday]","[52-Week, Wednesday]"
2,"[71, Biggest Movers, Friday]","[71, Biggest Movers, Friday]"
3,"[46, Friday, Mid-Day Session]","[46, Friday, Mid-Day Session]"
4,"[88, Maintains, Neutral, Price Target, Price T...","[88, Maintains, Neutral, Price Target, Price T..."


## 21. Build Entity Table

In [67]:
entity_table = (

    financial_df

    [

        [

            "news_id",

            "ticker",

            "published_date",

            "publisher",

            "headline",

            "normalized_entities",

            "entity_label",

            "entity_source"

        ]

    ]

    .explode(

        [

            "normalized_entities",

            "entity_label",

            "entity_source"

        ]

    )

    .rename(

        columns={

            "normalized_entities":"entity"

        }

    )

)

entity_table = entity_table.dropna(subset=["entity"])

print(entity_table.shape)

display(entity_table.head())

(9474461, 8)


,news_id,ticker,published_date,publisher,headline,entity,entity_label,entity_source
0,1,A,2020-06-05 14:30:54+00:00,Benzinga Insights,Stocks That Hit 52-Week Highs On Friday,52-Week,DATE,spaCy
0,1,A,2020-06-05 14:30:54+00:00,Benzinga Insights,Stocks That Hit 52-Week Highs On Friday,Friday,DATE,spaCy
1,2,A,2020-06-03 14:45:20+00:00,Benzinga Insights,Stocks That Hit 52-Week Highs On Wednesday,52-Week,DATE,spaCy
1,2,A,2020-06-03 14:45:20+00:00,Benzinga Insights,Stocks That Hit 52-Week Highs On Wednesday,Wednesday,DATE,spaCy
2,3,A,2020-05-26 08:30:07+00:00,Lisa Levin,71 Biggest Movers From Friday,71,CARDINAL,spaCy


## 22. Remove Duplicate Entities

In [68]:
entity_table = (

    entity_table

    .drop_duplicates(

        subset=[

            "news_id",

            "entity",

            "entity_label"

        ]

    )

    .reset_index(drop=True)

)

print(entity_table.shape)

(9471267, 8)


In [69]:
# normalized_entities = financial_df["normalized_entities"].copy()

In [70]:
# del financial_df
# gc.collect()

## 23. Entity Frequency

In [71]:
entity_frequency = (

    entity_table

    .groupby(

        [

            "entity",

            "entity_label"

        ]

    )

    .size()

    .reset_index(

        name="mentions"

    )

    .sort_values(

        "mentions",

        ascending=False

    )

)

display(entity_frequency.head(20))

,entity,entity_label,mentions
453910,Earnings,FINANCIAL_METRIC,283471
448174,EPS,FINANCIAL_METRIC,137183
876113,Stock,SECURITY,129981
335753,Buy,ANALYST_ACTION,113796
780690,Q4,QUARTER,95073
832378,Sales,FINANCIAL_METRIC,89942
780633,Q2,QUARTER,88419
436857,Dividend,CORPORATE_ACTION,88052
780598,Q1,QUARTER,85665
780660,Q3,QUARTER,84348


## 24. Entity Publisher Diversity

In [72]:
publisher_diversity = (

    entity_table

    .groupby("entity")

    ["publisher"]

    .nunique()

    .reset_index(

        name="publisher_diversity"

    )

)

## 25. Clean Entity Table

In [73]:
VALID_ENTITY_LABELS = {

    "ORG",
    "PERSON",
    "GPE",

    "FINANCIAL_METRIC",
    "CORPORATE_ACTION",
    "ANALYST_ACTION",
    "SECURITY",
    "MARKET_INDEX",
    "REGULATOR",
    "CRYPTO",
    "QUARTER",
    "CURRENCY",

    "MONEY",
    "REVENUE",
    "EPS",
    "EBITDA",
    "PRICE_TARGET"

}

entity_table = entity_table[
    entity_table["entity_label"].isin(
        VALID_ENTITY_LABELS
    )
].copy()

print(entity_table.shape)

(7614636, 8)


## 26. Remove Weak Entities

In [74]:
BAD_ENTITIES = {

    "",

    "Friday",
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Saturday",
    "Sunday",

    "Today",
    "Yesterday",
    "Tomorrow"

}

entity_table = entity_table[

    ~entity_table["entity"].isin(

        BAD_ENTITIES

    )

].copy()

## 27. Remove Pure Numbers

In [75]:
entity_table = entity_table[

    ~entity_table["entity"]

    .astype(str)

    .str.fullmatch(

        r"\d+"

    )

].copy()

print(entity_table.shape)

(7518748, 8)


## 28. Prefer Financial Labels

In [76]:
PRIORITY = {

    "FINANCIAL_METRIC": 1,
    "CORPORATE_ACTION": 2,
    "ANALYST_ACTION": 3,
    "SECURITY": 4,
    "MARKET_INDEX": 5,
    "REGULATOR": 6,
    "CRYPTO": 7,
    "QUARTER": 8,
    "CURRENCY": 9,
    "PRICE_TARGET": 10,
    "REVENUE": 11,
    "EPS": 12,
    "EBITDA": 13,

    "ORG": 20,
    "PERSON": 21,
    "GPE": 22,
    "MONEY": 23

}

entity_table["priority"] = (

    entity_table["entity_label"]

    .map(PRIORITY)

    .fillna(100)

)

entity_table = (

    entity_table

    .sort_values("priority")

    .drop_duplicates(

        subset=[

            "news_id",

            "entity"

        ],

        keep="first"

    )

    .drop(columns="priority")

)

print(entity_table.shape)

(7345743, 8)


## 29. Recompute Entity Frequency

In [77]:
entity_frequency = (

    entity_table

    .groupby(

        [

            "entity",

            "entity_label"

        ]

    )

    .size()

    .reset_index(

        name="mentions"

    )

    .sort_values(

        "mentions",

        ascending=False

    )

)

display(entity_frequency.head(25))

,entity,entity_label,mentions
397758,Earnings,FINANCIAL_METRIC,283471
392297,EPS,FINANCIAL_METRIC,137183
790156,Stock,SECURITY,129981
284680,Buy,ANALYST_ACTION,113796
700657,Q4,QUARTER,95073
749904,Sales,FINANCIAL_METRIC,89942
700611,Q2,QUARTER,88419
381304,Dividend,CORPORATE_ACTION,88052
700581,Q1,QUARTER,85665
700631,Q3,QUARTER,84348


## 30. Entity Popularity Score

In [78]:
publisher_diversity = (

    entity_table

    .groupby("entity")["publisher"]

    .nunique()

    .reset_index(name="publisher_diversity")

)

entity_popularity = (

    entity_frequency

    .merge(

        publisher_diversity,

        on="entity",

        how="left"

    )

)

entity_popularity["popularity_score"] = (

    np.log1p(entity_popularity["mentions"])

    *

    entity_popularity["publisher_diversity"]

)

entity_popularity = entity_popularity.sort_values(

    "popularity_score",

    ascending=False

)

display(entity_popularity.head(20))

,entity,entity_label,mentions,publisher_diversity,popularity_score
2,Stock,SECURITY,129981,332,3909.350218
0,Earnings,FINANCIAL_METRIC,283471,311,3904.564145
3,Buy,ANALYST_ACTION,113796,312,3632.357489
5,Sales,FINANCIAL_METRIC,89942,217,2475.304117
19,Outlook,FINANCIAL_METRIC,33105,212,2206.383601
16,China,GPE,35822,204,2139.214467
38,Dollar,CURRENCY,13155,225,2134.042472
25,Sell,ANALYST_ACTION,24524,208,2102.349243
7,Dividend,CORPORATE_ACTION,88052,175,1992.496482
1,EPS,FINANCIAL_METRIC,137183,158,1868.994382


## 31. Financial Entity Correction Dictionary

In [79]:
# Company names from ticker column

COMPANY_ENTITIES = set(

    financial_df["ticker"]

    .dropna()

    .astype(str)

    .str.upper()

)

# Publisher names

PUBLISHER_ENTITIES = set(

    financial_df["publisher"]

    .dropna()

    .astype(str)

)

# Major financial organizations

FINANCIAL_ORGANIZATIONS = {

    "Benzinga",

    "Reuters",

    "Bloomberg",

    "CNBC",

    "MarketWatch",

    "Yahoo Finance",

    "The Wall Street Journal",

    "Federal Reserve",

    "SEC",

    "NYSE",

    "NASDAQ",

    "S&P",

    "Dow Jones"

}

# Major exchanges

EXCHANGES = {

    "NASDAQ",

    "NYSE",

    "AMEX",

    "CBOE"

}

# Regulators

REGULATORS = {

    "Federal Reserve",

    "SEC",

    "FDA",

    "ECB",

    "FINRA",

    "DOJ"

}

print("Companies :", len(COMPANY_ENTITIES))
print("Publishers:", len(PUBLISHER_ENTITIES))

Companies : 6619
Publishers: 1047


## 32. Entity Label Correction Function

In [80]:
def correct_entity_label(entity, label):

    entity = str(entity).strip()

    upper = entity.upper()

    # Stock ticker

    if upper in COMPANY_ENTITIES:

        return "COMPANY"

    # Publisher

    if entity in PUBLISHER_ENTITIES:

        return "PUBLISHER"

    # Financial organizations

    if entity in FINANCIAL_ORGANIZATIONS:

        return "ORGANIZATION"

    # Exchanges

    if entity in EXCHANGES:

        return "EXCHANGE"

    # Regulators

    if entity in REGULATORS:

        return "REGULATOR"

    return label

## 33. Apply Corrections

In [81]:
entity_table["entity_label"] = (

    entity_table.apply(

        lambda row:

        correct_entity_label(

            row["entity"],

            row["entity_label"]

        ),

        axis=1

    )

)

display(

    entity_table.head()

)

,news_id,ticker,published_date,publisher,headline,entity,entity_label,entity_source
7577357,2369716,LII,2018-10-22 00:00:00+00:00,Zacks,Lennox International (LII) Q3 Earnings and Rev...,Earnings,FINANCIAL_METRIC,Matcher
3858112,1007293,PNFP,2020-01-21 00:00:00+00:00,Lisa Levin,"Earnings Scheduled For January 21, 2020",Earnings,FINANCIAL_METRIC,Matcher
2203235,575742,GRT,2012-07-06 00:00:00+00:00,David Johnson,UPDATE: Bank of America Downgrades Glimcher Re...,Sales,FINANCIAL_METRIC,Matcher
138,36,A,2020-03-26 00:00:00+00:00,Vick Meyer,Barclays Maintains Equal-Weight on Agilent Tec...,Price Target,FINANCIAL_METRIC,Matcher
63,14,A,2020-05-21 00:00:00+00:00,Benzinga Newsdesk,Agilent Technologies shares are trading higher...,EPS,COMPANY,Matcher


## 34. Rebuild Entity Frequency

In [82]:
entity_frequency = (

    entity_table

    .groupby(

        [

            "entity",

            "entity_label"

        ]

    )

    .size()

    .reset_index(

        name="mentions"

    )

    .sort_values(

        "mentions",

        ascending=False

    )

)

display(

    entity_frequency.head(30)

)

,entity,entity_label,mentions
397296,Earnings,FINANCIAL_METRIC,283471
391863,EPS,COMPANY,137188
788903,Stock,SECURITY,129981
284473,Buy,ANALYST_ACTION,113796
699590,Q4,QUARTER,95073
748660,Sales,FINANCIAL_METRIC,89942
699544,Q2,QUARTER,88419
380904,Dividend,CORPORATE_ACTION,88052
699514,Q1,QUARTER,85665
699564,Q3,QUARTER,84348


## 35. Recompute Entity Popularity

In [83]:
publisher_diversity = (

    entity_table

    .groupby("entity")["publisher"]

    .nunique()

    .reset_index(name="publisher_diversity")

)

entity_popularity = (

    entity_frequency

    .merge(

        publisher_diversity,

        on="entity",

        how="left"

    )

)

entity_popularity["popularity_score"] = (

    np.log1p(entity_popularity["mentions"])

    *

    entity_popularity["publisher_diversity"]

)

entity_popularity = entity_popularity.sort_values(

    "popularity_score",

    ascending=False

)

display(entity_popularity.head(20))

,entity,entity_label,mentions,publisher_diversity,popularity_score
2,Stock,SECURITY,129981,332,3909.350218
0,Earnings,FINANCIAL_METRIC,283471,311,3904.564145
3,Buy,ANALYST_ACTION,113796,312,3632.357489
5,Sales,FINANCIAL_METRIC,89942,217,2475.304117
19,Outlook,FINANCIAL_METRIC,33105,212,2206.383601
16,China,GPE,35822,204,2139.214467
38,Dollar,CURRENCY,13155,225,2134.042472
25,Sell,ANALYST_ACTION,24524,208,2102.349243
7,Dividend,CORPORATE_ACTION,88052,175,1992.496482
1,EPS,COMPANY,137188,158,1869.000141


## 36. Company -> Entity Relationships

In [84]:
chunk_size = 1000000

chunks = []

for start in range(0, len(entity_table), chunk_size):

    end = min(start + chunk_size, len(entity_table))

    chunk = entity_table.iloc[start:end]

    grouped = (

        chunk

        .groupby(

            [

                "ticker",

                "entity",

                "entity_label"

            ],

            observed=True

        )

        .size()

        .reset_index(name="mentions")

    )

    chunks.append(grouped)

company_entity_edges = (

    pd.concat(chunks, ignore_index=True)

    .groupby(

        [

            "ticker",

            "entity",

            "entity_label"

        ],

        observed=True

    )

    ["mentions"]

    .sum()

    .reset_index()

)

In [85]:
company_entity_edges["source"] = (

    "COMP_"

    + company_entity_edges["ticker"].astype(str)

)

company_entity_edges["target"] = (

    "ENTITY_"

    + company_entity_edges["entity"]

        .astype(str)

        .str.replace(" ", "_", regex=False)

)

company_entity_edges["relation"] = "MENTIONS"

company_entity_edges = company_entity_edges[
    [
        "source",
        "target",
        "relation",
        "mentions",
        "entity_label"
    ]
]

In [86]:
# company_entity_edges = (

#     entity_table

#     .groupby(

#         [

#             "ticker",

#             "entity",

#             "entity_label"

#         ]

#     )

#     .agg(

#         mentions=("entity", "count")

#     )

#     .reset_index()

# )

# company_entity_edges["source"] = (

#     "COMP_"

#     + company_entity_edges["ticker"].astype(str)

# )

# company_entity_edges["target"] = (

#     "ENTITY_"

#     + company_entity_edges["entity"]

#         .str.replace(" ", "_", regex=False)

# )

# company_entity_edges["relation"] = "MENTIONS"

# company_entity_edges = company_entity_edges[

#     [

#         "source",

#         "target",

#         "relation",

#         "mentions",

#         "entity_label"

#     ]

# ]

# print(company_entity_edges.shape)

# display(company_entity_edges.head())

## 37. Entity Co-occurrence Graph

In [87]:
from itertools import combinations

pairs = []

for entities in financial_df["normalized_entities"]:
# for entities in normalized_entities:

    if not isinstance(entities, list):
        continue

    entities = sorted(set(entities))

    if len(entities) < 2:
        continue

    pairs.extend(combinations(entities, 2))

entity_cooccurrence = (
    pd.DataFrame(
        pairs,
        columns=["entity_a", "entity_b"]
    )
    .value_counts()
    .reset_index(name="weight")
    .sort_values("weight", ascending=False)
)

print(entity_cooccurrence.shape)

display(entity_cooccurrence.head(20))

(7015613, 3)


,entity_a,entity_b,weight
0,EPS,Sales,48162
1,Earnings,Q4,40267
2,Earnings,Q2,37781
3,Earnings,Q3,35285
4,Earnings,Q1,34383
5,EPS,Q1,26275
6,EPS,Q4,26106
7,EPS,Q2,25654
8,EPS,Q3,25301
9,Maintains,Price Target,21376


## 38. Entity Timeline

In [88]:
entity_timeline = (

    entity_table

    .assign(

        event_date=pd.to_datetime(
            entity_table["published_date"]
        ).dt.date

    )

    .groupby(

        [

            "event_date",

            "entity",

            "entity_label"

        ]

    )

    .size()

    .reset_index(name="mentions")

)

print(entity_timeline.shape)

display(entity_timeline.head())

(3100630, 4)


,event_date,entity,entity_label,mentions
0,2009-04-29,Going Against the Herd,ORG,1
1,2009-05-22,Charles Sizemore Radio,PERSON,1
2,2009-05-27,$15,MONEY,3
3,2009-05-27,$20,MONEY,3
4,2009-05-27,JVA,COMPANY,6


## 39. Entity Statistics

In [89]:
entity_statistics = (

    entity_table

    .groupby(

        [

            "entity",

            "entity_label"

        ]

    )

    .agg(

        total_mentions=("entity","count"),

        companies=("ticker","nunique"),

        publishers=("publisher","nunique"),

        first_seen=("published_date","min"),

        last_seen=("published_date","max")

    )

    .reset_index()

)

display(

    entity_statistics.head()

)

,entity,entity_label,total_mentions,companies,publishers,first_seen,last_seen
0,"""European Union",ORG,1,1,1,2015-01-13 00:00:00+00:00,2015-01-13 00:00:00+00:00
1,"""News Corp.",ORG,1,1,1,2011-05-06 00:00:00+00:00,2011-05-06 00:00:00+00:00
2,# 5 Additions,MONEY,6,6,1,2013-01-18 00:00:00+00:00,2013-01-22 00:00:00+00:00
3,#1,MONEY,18,18,3,2012-10-15 00:00:00+00:00,2018-09-26 00:00:00+00:00
4,#1 Addition,MONEY,167,101,1,2013-01-28 00:00:00+00:00,2013-04-30 00:00:00+00:00


## 40. Entity Lookup Dictionary

In [90]:
# entity_lookup = {

#     entity: group

#     for entity, group in

#     entity_table.groupby("entity")

# }
entity_lookup = (
    entity_table
    .groupby("entity")["news_id"]
    .apply(list)
    .to_dict()
)

print("Entity Lookup Size :", len(entity_lookup))

Entity Lookup Size : 902292


## 41. Save Everything

In [91]:
entity_table.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "news_entities.parquet"

    ),

    index=False

)

entity_frequency.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "entity_frequency.parquet"

    ),

    index=False

)

entity_popularity.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "entity_popularity.parquet"

    ),

    index=False

)

entity_statistics.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "entity_statistics.parquet"

    ),

    index=False

)

company_entity_edges.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "company_entity_edges.parquet"

    ),

    index=False

)

entity_cooccurrence.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "entity_cooccurrence.parquet"

    ),

    index=False

)

entity_timeline.to_parquet(

    os.path.join(

        ENTITY_PATH,

        "entity_timeline.parquet"

    ),

    index=False

)

with open(

    os.path.join(

        ENTITY_PATH,

        "entity_lookup.pkl"

    ),

    "wb"

) as f:

    pickle.dump(

        entity_lookup,

        f

    )

print("Entity Intelligence datasets saved successfully.")

Entity Intelligence datasets saved successfully.


## 42. Evaluation

In [92]:
evaluation = pd.DataFrame({

    "Metric":[

        "News Articles",

        "Entity Records",

        "Unique Entities",

        "Entity Types",

        "Companies",

        "Publishers",

        "Company-Entity Edges",

        "Entity Co-occurrences",

        "Timeline Records"

    ],

    "Value":[

        len(financial_df),

        len(entity_table),

        entity_table["entity"].nunique(),

        entity_table["entity_label"].nunique(),

        entity_table["ticker"].nunique(),

        entity_table["publisher"].nunique(),

        len(company_entity_edges),

        len(entity_cooccurrence),

        len(entity_timeline)

    ]

})

display(evaluation)

,Metric,Value
0,News Articles,3215296
1,Entity Records,7345743
2,Unique Entities,902292
3,Entity Types,21
4,Companies,6597
5,Publishers,918
6,Company-Entity Edges,3457755
7,Entity Co-occurrences,7015613
8,Timeline Records,3100630


## 43. Display Top Entities

In [93]:
display(

    entity_popularity.head(25)

)

,entity,entity_label,mentions,publisher_diversity,popularity_score
2,Stock,SECURITY,129981,332,3909.350218
0,Earnings,FINANCIAL_METRIC,283471,311,3904.564145
3,Buy,ANALYST_ACTION,113796,312,3632.357489
5,Sales,FINANCIAL_METRIC,89942,217,2475.304117
19,Outlook,FINANCIAL_METRIC,33105,212,2206.383601
16,China,GPE,35822,204,2139.214467
38,Dollar,CURRENCY,13155,225,2134.042472
25,Sell,ANALYST_ACTION,24524,208,2102.349243
7,Dividend,CORPORATE_ACTION,88052,175,1992.496482
1,EPS,COMPANY,137188,158,1869.000141


## 45. Summary

In [95]:
summary = pd.DataFrame({

    "Output":[

        "news_entities.parquet",

        "entity_frequency.parquet",

        "entity_popularity.parquet",

        "entity_statistics.parquet",

        "company_entity_edges.parquet",

        "entity_cooccurrence.parquet",

        "entity_timeline.parquet",

        "entity_lookup.pkl"

    ]

})

display(summary)


,Output
0,news_entities.parquet
1,entity_frequency.parquet
2,entity_popularity.parquet
3,entity_statistics.parquet
4,company_entity_edges.parquet
5,entity_cooccurrence.parquet
6,entity_timeline.parquet
7,entity_lookup.pkl
